# C4 — Notebook 01: shared data contract and spatial QA

**Purpose:** Validate the **real 1,429 DMC labelled station-days**, **809/269/351 date splits**, the **private 40-point gauge registry**, and the **39 upstream catchment polygons used by labelled stations**. If the 40th polygon exists, inspect it as well. Do **not** rerun MERIT or download imagery.

**Forecast convention:** Original source contains a **nominal 09:30 public report time**. This notebook adds a distinct **simulated 09:00 historical hindcast origin**, not proof that the 09:00 stage was publicly available then. Stage labels are next-day 09:00 observations. No inputs from the future are assembled here.

In [ ]:
# Cell 1 — Imports and repository discovery 
from pathlib import Path
from collections import Counter
from datetime import timedelta
import hashlib, json, re, warnings
import numpy as np
import pandas as pd
from shapely.geometry import shape, Point

# OPTIONAL: set manually if VS Code starts outside the repository.
REPO_ROOT_OVERRIDE = None  

def find_repo_root():
    if REPO_ROOT_OVERRIDE:
        p = Path(REPO_ROOT_OVERRIDE).expanduser().resolve()
        if not (p/'c4_fusion').is_dir() or not (p/'shared').is_dir():
            raise RuntimeError(f"Not a FloodRisk repository: {p}")
        return p
    for start in [Path.cwd().resolve(), Path.cwd().resolve().parent]:
        for candidate in [start, *start.parents]:
            if (candidate/'c4_fusion').is_dir() and (candidate/'shared'/'spatial'/'catchments').is_dir():
                return candidate
    raise FileNotFoundError('Repository root not found; set REPO_ROOT_OVERRIDE above to the J26-DS-314 directory.')

ROOT = find_repo_root()
PRIVATE_GAUGES = ROOT/'c4_fusion'/'private_data'/'stations_40_PRIVATE.geojson'
PRIVATE_LABELS = ROOT/'c4_fusion'/'private_data'/'observations'/'FloodRisk_DMC_StationDay_LABELLED_1429.csv'
SHARED_LABELS = ROOT/'shared'/'observations'/'dmc_2025nov17_dec26'/'FloodRisk_DMC_StationDay_LABELLED_1429.csv'
PRIVATE_MASTER = ROOT/'c4_fusion'/'private_data'/'observations'/'FloodRisk_DMC_StationDay_MASTER_1600.csv'
SHARED_MASTER = ROOT/'shared'/'observations'/'dmc_2025nov17_dec26'/'FloodRisk_DMC_StationDay_MASTER_1600.csv'
CATCHMENTS_ROOT = ROOT/'shared'/'spatial'/'catchments'
OUT_DIR = ROOT/'data'/'processed'/'c4'/'manifests'

LABELS_CSV = next((p for p in [PRIVATE_LABELS,SHARED_LABELS] if p.is_file()), PRIVATE_LABELS)
MASTER_CSV = next((p for p in [PRIVATE_MASTER,SHARED_MASTER] if p.is_file()), None)
for name, p in [('private stations',PRIVATE_GAUGES),('labelled DMC CSV',LABELS_CSV)]:
    if not p.is_file():
        raise FileNotFoundError(f'Missing {name}: {p}. See the Setup note below. No data were modified.')
if LABELS_CSV == SHARED_LABELS:
    print('CAUTION: Labelled CSV is in shared/; keep this source-derived file Git-ignored if redistribution permission is not established.')
print('Repository:', ROOT)
print('Private station registry and labelled DMC data found.')
print('Master 1600-row source:', 'present' if MASTER_CSV else 'optional / not found')

Repository: C:\Users\ASUS\projects\rp_project\J26-DS-314
Private station registry and labelled DMC data found.
Master 1600-row source: present


In [2]:
# Cell 2 — Load and validate the canonical private station registry

def station_slug(value):
    return re.sub(r'_+','_',re.sub(r'[^a-z0-9]+','_',str(value).casefold())).strip('_')

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def local_repo_path(value):
    """CSV repo-relative path: prevent outside-repository traversal."""
    p = (ROOT / str(value).replace('\\','/')).resolve()
    try:
        p.relative_to(ROOT)
    except ValueError as exc:
        raise ValueError(f'CSV path outside repo: {value!r}') from exc
    return p

geo = json.loads(PRIVATE_GAUGES.read_text(encoding='utf-8'))
assert geo.get('type')=='FeatureCollection', 'Private station file must be GeoJSON FeatureCollection.'
assert len(geo.get('features',[]))==40, 'Private registry must contain EXACTLY 40 research gauges.'
station_features = {}
for f in geo['features']:
    assert f.get('type')=='Feature' and f['geometry']['type']=='Point'
    sid=f['properties']['station_id']
    assert sid == station_slug(f['properties']['station_name'])
    assert sid not in station_features, f'Duplicate station_id {sid}'
    lon,lat=f['geometry']['coordinates']
    assert -180<=lon<=180 and -90<=lat<=90 and 5<=lat<=10 and 79<=lon<=82, f'Unreasonable WGS84 point: {sid}'
    station_features[sid]=f
assert not {'ampara','calidonia'}.intersection(station_features), 'Two unused stations must be excluded.'
assert 'thambuththegama' in station_features, '40th permanent gauge must be preserved.'
print('PASS: 40 unique private WGS84 research gauge locations; Ampara/Calidonia excluded.')
print('Source attribute coordinate issues flagged:',[k for k,v in station_features.items() if v['properties'].get('coordinate_properties_disagree_with_geometry')])

PASS: 40 unique private WGS84 research gauge locations; Ampara/Calidonia excluded.
Source attribute coordinate issues flagged: ['moragaswewa']


In [3]:
# Cell 3 — Load numerical label pairs, test identities, units and chronological splits
EXPECTED_SPLITS={'TRAIN':809,'VALIDATION':269,'TEST':351}
START=pd.Timestamp('2025-11-17')
END=pd.Timestamp('2025-12-26')
EXPECTED_COLUMNS={
 'record_id','station_id','station_name','basin','forecast_date','chronological_split',
 'stage_at_issue_m','target_stage_next_day_m','target_delta_stage_m','stage_label_valid',
 'stage_0900_obs_time_local','target_0900_obs_time_local','issue_time_assumed_local',
 'catchment_polygon_path','gauge_lon_wgs84_arcgis','gauge_lat_wgs84_arcgis','source_unit'
}
labels=pd.read_csv(LABELS_CSV,encoding='utf-8-sig',low_memory=False)
missing_columns=sorted(EXPECTED_COLUMNS-set(labels.columns))
assert not missing_columns, f'Labelled CSV missing required columns: {missing_columns}'
assert len(labels)==1429, f'Expected 1429 source-derived labels; received {len(labels)}.'
assert labels['record_id'].notna().all() and labels['record_id'].is_unique,'Duplicated or missing record_id.'
assert not labels[['station_id','forecast_date']].duplicated().any(), 'Repeated station + forecast_date.'
assert labels['station_id'].nunique()==39, 'Expected 39 distinct gauges with DMC target pairs.'
assert set(labels['station_id'])==set(station_features)-{'thambuththegama'}, 'DMC IDs do not match the private 40-station registry.'
assert labels['station_id'].map(station_slug).eq(labels['station_id']).all(), 'Noncanonical station ID.'
assert (labels['record_id']==labels['station_id']+'_'+labels['forecast_date']).all(), 'record_id pattern inconsistent.'
assert labels['stage_label_valid'].astype(str).str.lower().eq('true').all(), 'Some labelled rows are not marked valid.'
actual_counts=labels['chronological_split'].value_counts().to_dict()
assert actual_counts==EXPECTED_SPLITS, f'Split counts changed: {actual_counts}'
labels['forecast_dt']=pd.to_datetime(labels['forecast_date'],format='%Y-%m-%d',errors='raise')
assert labels['forecast_dt'].between(START, END-pd.Timedelta(days=1)).all(), 'Unexpected forecast date.'

periods={'TRAIN':('2025-11-17','2025-12-09'), 'VALIDATION':('2025-12-10','2025-12-16'), 'TEST':('2025-12-17','2025-12-25')}
for split,(first,last) in periods.items():
    one=labels[labels['chronological_split']==split]
    assert one['forecast_dt'].between(first,last).all(),f'{split} dates outside frozen boundaries.'

for key in ['stage_at_issue_m','target_stage_next_day_m','target_delta_stage_m']:
    labels[key]=pd.to_numeric(labels[key],errors='coerce')
    assert np.isfinite(labels[key].to_numpy(dtype=float)).all(),f'Non-numeric or infinite {key}.'
assert np.allclose((labels['target_stage_next_day_m']-labels['stage_at_issue_m']),labels['target_delta_stage_m'],atol=1e-5), 'Stage-change arithmetic does not match target/current levels.'
assert labels['source_unit'].isin(['m','ft']).all(), 'Unexpected source units.'
assert set(labels.loc[labels['source_unit']=='ft','station_id'])=={'nagalagam_street'}, 'Foot-to-metre source-unit mapping changed.'
if {'stage_0900_original','target_0900_next_day_original'}.issubset(labels):
    factor=np.where(labels['source_unit'].eq('ft'),0.3048,1.0)
    for orig,metres in [('stage_0900_original','stage_at_issue_m'),('target_0900_next_day_original','target_stage_next_day_m')]:
        values=pd.to_numeric(labels[orig],errors='coerce')
        assert values.notna().all()
        assert np.allclose(values*factor,labels[metres],atol=1e-4),f'Unit conversion inconsistent: {orig}.'
print('PASS: 1429 unique numeric pairs; 39 labelled gauges; 809 train / 269 validation / 351 test.')
print('PASS: metres and feet conversion; target = next-day level; delta = next minus current.')

PASS: 1429 unique numeric pairs; 39 labelled gauges; 809 train / 269 validation / 351 test.
PASS: metres and feet conversion; target = next-day level; delta = next minus current.


In [4]:
# Cell 4 — Verify the observation/target times; make the 09:00 hindcast assumption explicit

def parse_source_clock(values,clock):
    txt=values.astype(str)
    assert txt.str.endswith('Asia/Colombo').all(), f'Missing/changed timezone in {values.name}'
    out=pd.to_datetime(txt.str[:16],format='%Y-%m-%d %H:%M',errors='raise')
    assert (out.dt.hour==clock[0]).all() and (out.dt.minute==clock[1]).all(),f'Clock mismatch in {values.name}'
    return out

obs=parse_source_clock(labels['stage_0900_obs_time_local'],(9,0))
target_obs=parse_source_clock(labels['target_0900_obs_time_local'],(9,0))
report=parse_source_clock(labels['issue_time_assumed_local'],(9,30))
assert obs.eq(labels['forecast_dt']+pd.Timedelta(hours=9)).all()
assert target_obs.eq(obs+pd.Timedelta(days=1)).all()
assert report.eq(obs+pd.Timedelta(minutes=30)).all()
# A SIMULATED 09:00 experiment; the 09:30 nominal publication record stays untouched.
labels['hindcast_origin_0900_local']=obs.dt.strftime('%Y-%m-%d %H:%M Asia/Colombo')
labels['target_time_0900_local']=target_obs.dt.strftime('%Y-%m-%d %H:%M Asia/Colombo')
assert labels['target_time_0900_local'].str[:10].eq((labels['forecast_dt']+pd.Timedelta(days=1)).dt.strftime('%Y-%m-%d')).all()
print('PASS: observation-to-observation horizon = 24 hours.')
print('CAUTION: hindcast origin T 09:00 is simulated; original nominal report is T 09:30.')

PASS: observation-to-observation horizon = 24 hours.
CAUTION: hindcast origin T 09:00 is simulated; original nominal report is T 09:30.


In [5]:
# Cell 5 — Optional 1600-row master reconciliation (never replace or interpolate missing stages)
if MASTER_CSV:
    master=pd.read_csv(MASTER_CSV,encoding='utf-8-sig',low_memory=False)
    assert len(master)==1600 and master['record_id'].is_unique, 'Master expected 1600 unique rows.'
    assert master['station_id'].nunique()==40, 'Master must contain all 40 permanent station IDs.'
    assert set(master['station_id'])==set(station_features), 'Master station registry differs from private registry.'
    assert set(labels['record_id']).issubset(set(master['record_id'])), 'A labelled record is absent from master.'
    merged=labels.merge(master[['record_id','target_stage_next_day_m','stage_at_issue_m']],on='record_id',how='left',validate='one_to_one',suffixes=('','_master'))
    for key in ['target_stage_next_day_m','stage_at_issue_m']:
        assert np.allclose(merged[key],pd.to_numeric(merged[key+'_master'],errors='coerce'),atol=1e-5), f'Master mismatch in {key}.'
    print('PASS: Optional master confirms 1600 canonical slots and all 1429 labelled pairs.')
else:
    print('NOTICE: 1600-row master not present; 40 station identities checked against private registry instead.')

PASS: Optional master confirms 1600 canonical slots and all 1429 labelled pairs.


In [6]:
# Cell 6 — Match every labelled station to its published candidate polygon (read-only)
# Shapes are real GeoJSON geometry, NOT bounding boxes or simplified QA HTML outlines.
summary=[]
polygon_files={}
for sid, group in labels.groupby('station_id',sort=True):
    file_names=group['catchment_polygon_path'].dropna().unique()
    assert len(file_names)==1, f'Conflicting polygon paths for station {sid}: {file_names}'
    rel=str(file_names[0]).replace('\\','/')
    assert rel.startswith('shared/spatial/catchments/'),f'Unrecognized catchment source path for {sid}'
    assert rel.endswith('_catchment_polygon_CANDIDATE.geojson'),f'Not a candidate polygon: {sid}'
    path=local_repo_path(rel)
    if not path.is_file():
        raise FileNotFoundError(f'Missing catchment polygon for {sid}: {path} — merge your spatial-data PR or copy the existing polygon; do not regenerate it.')
    doc=json.loads(path.read_text(encoding='utf-8'))
    feats=doc['features'] if doc.get('type')=='FeatureCollection' else [doc]
    assert len(feats)==1, f'{sid} polygon expected exactly one feature.'
    feature=feats[0]
    props=feature.get('properties',{})
    claimed=props.get('station') or props.get('station_name')
    if claimed is not None:
        assert station_slug(claimed)==sid, f'Polygon station identity mismatch: {sid} vs {claimed}'
    geom=shape(feature['geometry'])
    assert geom.geom_type in ('Polygon','MultiPolygon') and geom.is_valid and not geom.is_empty and geom.area>0, f'Invalid catchment polygon: {sid}'
    basin=station_features[sid]['properties']['basin']
    assert group['basin'].astype(str).str.strip().eq(basin).all(), f'Basin mismatch in labelled CSV: {sid}'
    src_lon,src_lat=station_features[sid]['geometry']['coordinates']
    assert np.allclose(pd.to_numeric(group['gauge_lon_wgs84_arcgis']),src_lon,atol=1e-6,rtol=0),f'Gauge longitude disagrees with source GeoJSON: {sid}'
    assert np.allclose(pd.to_numeric(group['gauge_lat_wgs84_arcgis']),src_lat,atol=1e-6,rtol=0),f'Gauge latitude disagrees with source GeoJSON: {sid}'
    current=props.get('outlet_lon'), props.get('outlet_lat')
    if all(v is not None for v in current):
        assert geom.buffer(1e-10).covers(Point(float(current[0]),float(current[1]))),f'Computational outlet is outside its real polygon: {sid}'
    box_path=path.parent.parent/'bounding_boxes'/path.name.replace('_catchment_polygon_CANDIDATE.geojson','_bounding_box_CANDIDATE.geojson')
    bbox_status='FILE_NOT_PRESENT'
    if box_path.is_file():
        boxdoc=json.loads(box_path.read_text(encoding='utf-8'))
        boxfeatures=boxdoc['features'] if boxdoc.get('type')=='FeatureCollection' else [boxdoc]
        if len(boxfeatures)==1:
            bboxgeom=shape(boxfeatures[0]['geometry'])
            assert bboxgeom.buffer(1e-8).covers(geom),f'Bounding box does not contain polygon: {sid}'
            bbox_status='COVERS_POLYGON'
        else:
            bbox_status='UNEXPECTED_FEATURE_COUNT'
    qas=group['catchment_qa_status'].dropna().unique()
    assert len(qas)==1, f'Nonconstant catchment QA status for {sid}.'
    summary.append({'station_id':sid,'station_name':group.iloc[0]['station_name'],'basin':basin,
        'labelled_rows':len(group),'polygon_repo_path':rel,'polygon_sha256':sha256(path),
        'polygon_valid':True,'outlet_checked':all(v is not None for v in current),
        'original_gauge_inside_polygon':bool(geom.covers(Point(src_lon,src_lat))),
        'bounding_box_status':bbox_status,'catchment_qa_status':str(qas[0]),
        'source_coordinate_warning':bool(station_features[sid]['properties'].get('coordinate_properties_disagree_with_geometry'))})
    polygon_files[sid]=path
qa=pd.DataFrame(summary)
assert len(qa)==39 and qa['station_id'].is_unique
print('PASS: All 39 labelled stations have distinct existing, valid candidate upstream polygons.')
print('Original gauges outside candidate polygon (diagnostic, not automatic error):',int((~qa['original_gauge_inside_polygon']).sum()))
print('QA flags retained:',qa['catchment_qa_status'].value_counts().to_dict())

PASS: All 39 labelled stations have distinct existing, valid candidate upstream polygons.
Original gauges outside candidate polygon (diagnostic, not automatic error): 3
QA flags retained: {'CANDIDATE_PENDING_VISUAL_QA': 35, 'REVIEW_OFFICIAL_DRAINAGE_DIFFERENCE_GT20': 2, 'REVIEW_SOURCE_COORDINATE_DISCREPANCY': 1, 'REVIEW_DISTANT_OUTLET': 1}


In [8]:

outside = qa.loc[
    ~qa["original_gauge_inside_polygon"],
    ["station_name", "basin", "catchment_qa_status"]
]

print("Original gauges outside candidate polygons:")
display(outside)

Original gauges outside candidate polygons:


,station_name,basin,catchment_qa_status
4,Ellagawa,Kalu Ganga,CANDIDATE_PENDING_VISUAL_QA
26,Pitabeddara,Nilwala Ganga,CANDIDATE_PENDING_VISUAL_QA
27,Putupaula,Kalu Ganga,CANDIDATE_PENDING_VISUAL_QA


In [9]:

from shapely.geometry import Point, shape
from shapely.ops import nearest_points
from pyproj import Geod
import json

geod = Geod(ellps="WGS84")

for _, row in outside.iterrows():
    sid = qa.loc[
        qa["station_name"] == row["station_name"],
        "station_id"
    ].iloc[0]

    path = polygon_files[sid]
    doc = json.loads(path.read_text(encoding="utf-8"))
    feature = doc["features"][0]
    geom = shape(feature["geometry"])

    lon, lat = station_features[sid]["geometry"]["coordinates"]
    gauge = Point(lon, lat)

    nearest = nearest_points(gauge, geom)[1]
    distance_m = geod.inv(
        gauge.x, gauge.y, nearest.x, nearest.y
    )[2]

    props = feature.get("properties", {})
    outlet_distance = props.get("distance_m", props.get("outlet_distance_m"))

    print(
        f"{row['station_name']}: "
        f"{distance_m:.1f} m outside polygon; "
        f"outlet displacement metadata={outlet_distance}"
    )


Ellagawa: 33.6 m outside polygon; outlet displacement metadata=80.18
Pitabeddara: 0.4 m outside polygon; outlet displacement metadata=47.03
Putupaula: 28.7 m outside polygon; outlet displacement metadata=76.71


In [10]:
# Cell 7 — Inspect the 40th station polygon if present (no DMC observations or labels)
extra_sid='thambuththegama'
master_row=None
if MASTER_CSV:
    mr=master.loc[master['station_id']==extra_sid]
    if len(mr):
        master_row=mr.iloc[0]
if master_row is not None:
    expected_rel=str(master_row['catchment_polygon_path']).replace('\\','/')
else:
    expected_rel='shared/spatial/catchments/kala_oya/polygons/thambuththegama_catchment_polygon_CANDIDATE.geojson'
extra_path=local_repo_path(expected_rel)
if extra_path.is_file():
    obj=json.loads(extra_path.read_text(encoding='utf-8'))
    arr=obj.get('features',[]) if obj.get('type')=='FeatureCollection' else [obj]
    assert len(arr)==1 and shape(arr[0]['geometry']).is_valid
    extra_result='PRESENT_VALID_NO_DMC_LABELS'
else:
    extra_result='NO_LABELS_POLYGON_NOT_FOUND_OPTIONAL'
print('40th station:', extra_sid, extra_result)

40th station: thambuththegama PRESENT_VALID_NO_DMC_LABELS


40 candidate catchments were **generated and technically screened**, none is independently certified as an exact upstream watershed.

In [11]:
# Cell 8 — Build a future-target-free sample index and a private supervised-label table
# Downstream *retrieval* notebooks consume INDEX, never future target values.
index_columns=['record_id','station_id','station_name','basin','forecast_date','chronological_split',
    'hindcast_origin_0900_local','target_time_0900_local','catchment_polygon_path','catchment_qa_status']
index=labels[index_columns].copy().rename(columns={'chronological_split':'split','catchment_polygon_path':'polygon_repo_path'})
index['station_geojson_private_path']='c4_fusion/private_data/stations_40_PRIVATE.geojson'
index['sar_ref']=''        # Populated in Notebook 02
index['swi_ref']=''        # Populated in Notebook 03
index['weather_ref']=''    # Populated in Notebook 04
index['environment_ref']='' # Populated in Notebook 05
index=index.sort_values(['forecast_date','station_id']).reset_index(drop=True)
assert not {'target_stage_next_day_m','target_delta_stage_m'}.intersection(index.columns), 'Future targets leaked to feature retrieval index.'
training=labels[['record_id','station_id','forecast_date','chronological_split','stage_at_issue_m',
    'target_stage_next_day_m','target_delta_stage_m','stage_0900_obs_time_local','target_0900_obs_time_local',
    'current_source_excel_row','next_day_source_excel_row','source_unit','catchment_qa_status']].copy()
training=training.rename(columns={'chronological_split':'split'}).sort_values(['forecast_date','station_id']).reset_index(drop=True)
assert len(index)==len(training)==1429
assert index['record_id'].tolist()==training['record_id'].tolist()
print('PASS: Target-free source-retrieval index and separate supervised training-label table prepared.')

PASS: Target-free source-retrieval index and separate supervised training-label table prepared.


In [12]:
# Cell 9 — Atomic, idempotent saving with SHA256 provenance; no public coordinate tables
from os import replace

def save_if_different(destination, data):
    destination=Path(destination)
    destination.parent.mkdir(parents=True,exist_ok=True)
    if destination.is_file() and destination.read_bytes()==data:
        return 'UNCHANGED'
    temp=destination.with_name(destination.name+'.writing.tmp')
    temp.write_bytes(data)
    replace(temp,destination)
    return 'WRITTEN'

sources=[('DMC_LABELS',LABELS_CSV),('PRIVATE_GAUGE_GEOJSON',PRIVATE_GAUGES)]
if MASTER_CSV: sources.append(('DMC_MASTER',MASTER_CSV))
sources += [(f'CATCHMENT_{k}',v) for k,v in sorted(polygon_files.items())]
source_hashes=pd.DataFrame([{'source_type':name,'repo_or_private_path':str(path.relative_to(ROOT)),'sha256':sha256(path)} for name,path in sources])

full_registry=[]
for sid,feature in sorted(station_features.items()):
    q=qa.set_index('station_id')
    full_registry.append({'station_id':sid,'station_name':feature['properties']['station_name'],
      'basin':feature['properties']['basin'],'has_labelled_pairs':sid in polygon_files,
      'n_labelled_pairs':int(q.loc[sid,'labelled_rows']) if sid in q.index else 0,
      'polygon_qa_status':str(q.loc[sid,'catchment_qa_status']) if sid in q.index else 'NO_DMC_LABELS',
      'source_coordinate_warning':bool(feature['properties']['coordinate_properties_disagree_with_geometry']),
      'private_geojson_path':'c4_fusion/private_data/stations_40_PRIVATE.geojson'})
reg=pd.DataFrame(full_registry)

report={
  'project':'FloodRisk Sri Lanka — independent C4 retrospective hindcast',
  'notebook':'01_shared_contract_and_spatial_QA.ipynb',
  'source_label_rows':len(labels),'unique_labelled_gauges':labels.station_id.nunique(),
  'private_station_points':len(station_features),'master_rows':int(len(master)) if MASTER_CSV else None,
  'chronological_split_counts':actual_counts,
  'forecast_origin':'SIMULATED Day T 09:00 Asia/Colombo (not verified public availability)',
  'original_nominal_report_time':'Day T 09:30 Asia/Colombo',
  'target_time':'Day T+1 09:00 Asia/Colombo',
  'prediction_target':'Delta stage in metres; predicted next-day stage = stage T + predicted delta',
  'polygon_validation':'39 required labelled polygons valid; 40th optional as no labels',
  'optional_40th_polygon':extra_result,
  'source_station_coordinate_conflicts':[k for k,v in station_features.items() if v['properties'].get('coordinate_properties_disagree_with_geometry')],
  'catchment_flags':{k:int(v) for k,v in qa.catchment_qa_status.value_counts().items()},
  'public_release_warning':'Do not publish numerical source data or private gauge GeoJSON without permission. Outputs contain research-derived station identities but no exact coordinate columns.',
  'input_source_sha256':{key:sha256(path) for key,path in sources}
}

out={
 'c4_sample_index_1429.csv':index.to_csv(index=False).encode('utf-8-sig'),
 'c4_training_labels_1429_PRIVATE.csv':training.to_csv(index=False).encode('utf-8-sig'),
 'c4_spatial_qa_39.csv':qa.to_csv(index=False).encode('utf-8-sig'),
 'c4_station_registry_40_no_coordinates.csv':reg.to_csv(index=False).encode('utf-8-sig'),
 'c4_source_hashes.csv':source_hashes.to_csv(index=False).encode('utf-8-sig'),
 'c4_notebook01_audit.json':(json.dumps(report,indent=2,ensure_ascii=False,sort_keys=True)+'\n').encode('utf-8'),
}
for filename,content in out.items():
    print(f'{filename}: {save_if_different(OUT_DIR/filename,content)}')
assert (OUT_DIR/'c4_sample_index_1429.csv').is_file()
print('PASSED: all mandatory Notebook 01 checks; outputs in:', OUT_DIR)

c4_sample_index_1429.csv: WRITTEN
c4_training_labels_1429_PRIVATE.csv: WRITTEN
c4_spatial_qa_39.csv: WRITTEN
c4_station_registry_40_no_coordinates.csv: WRITTEN
c4_source_hashes.csv: WRITTEN
c4_notebook01_audit.json: WRITTEN
PASSED: all mandatory Notebook 01 checks; outputs in: C:\Users\ASUS\projects\rp_project\J26-DS-314\data\processed\c4\manifests


## Interpretation

**Passed here:** source-derived numerical labels, unique identities, fixed chronological split, source-unit conversion, 24-hour measurement horizon, private WGS84 station-to-catchment matching, polygon validity and output provenance. **Not checked here:** whether a candidate polygon is the true upstream watershed, whether 09:00 observations were actually published by 09:00, or whether all satellite/weather sources are available.